# Method 1: XTTS-v2 (zero-shot voice cloning)

**Nothing is trained in this notebook.** The model listens to a short reference clip of the teacher and then speaks your 20 sentences in that voice.

## What you upload
- **ONE audio or video file of the teacher** (mp3, wav, m4a or mp4 all work).
- Only the teacher's voice in the part you use: no music, no students, no other voices.
- At least about 6 seconds of clear speech. 10 to 20 seconds is plenty. A longer file is fine: you pick which stretch to use in the config cell.
- **No transcript needed** for this method.
- You do **not** need to convert it to wav yourself. The notebook converts it.

## What you get
- 20 audio files `00.wav` to `19.wav` (one per sentence) and a zip `xtts_results.zip` that downloads at the end.
- Keep the zip. The separate compare notebook (`4_compare.ipynb`) uses it.

## Steps
1. `Runtime > Change runtime type > T4 GPU`.
2. Run the cells from top to bottom. Step 3 asks you to upload the file.
3. After Step 3, **play the reference clip** to confirm it is only the teacher.

## Step 1: Install (run this first, before anything else)

In [ ]:
!pip -q install coqui-tts soundfile

If Colab shows a message asking to restart the runtime, click **Restart session**, then continue from Step 2.

## Step 2: Check the GPU and edit the config

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ===================== CONFIG: edit only this cell =====================
REF_START_SEC = 0      # where in your file the reference clip starts (seconds). Pick a stretch where ONLY the teacher speaks.
REF_SECONDS   = 15     # length of the reference clip (10 to 20 is good)

# XTTS-v2 weights use the Coqui Public Model License (non-commercial). Read it, then set this to True.
ACCEPT_COQUI_LICENSE = False

# The 20 sentences below are the same in all three method notebooks, so the results can be compared. If you change them in one notebook, change them identically in the others.
SENTENCES = [
    "Good morning everyone, let us begin with a quick recap of the previous lecture.",
    "Please open your notebooks and follow along with the example on the screen.",
    "Today we will look at how the method works and why it matters in practice.",
    "If you have any doubts, please ask them at the end of this section.",
    "The assignment is due next week, so please start working on it early.",
    "Let us go through the main steps one by one.",
    "This is a common mistake that students make during the exam.",
    "Remember that the goal is to understand the idea, not just memorize the formula.",
    "We will now move on to the next topic.",
    "To summarize, there are three key points you should take away from this lecture.",
    "A convolution slides a small kernel across the image to produce a feature map.",
    "Histogram equalization improves the contrast of an image.",
    "The Canny edge detector uses two thresholds for hysteresis.",
    "Backpropagation computes gradients using the chain rule.",
    "Gradient descent updates the weights in the direction that reduces the loss.",
    "A convolutional neural network applies pooling to reduce the spatial resolution.",
    "The ResNet architecture uses skip connections to ease the training of deep networks.",
    "Overfitting happens when the model memorizes the training data.",
    "Precision and recall describe different kinds of errors in a classifier.",
    "SIFT detects keypoints that are invariant to scale and rotation.",
]
# --- end of config ---

## Step 3: Upload your teacher file (it is converted automatically)

In [ ]:
import os, sys, shutil, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("Command failed: " + cmd)
    return r.stdout

import soundfile as sf
METHOD = "xtts"
WORK = "/content/xtts"
OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)

from google.colab import files
print("Click 'Choose files' and select ONE recording of the teacher.")
up = files.upload()
name = next(iter(up))
SRC = f"{WORK}/teacher_source{os.path.splitext(name)[1]}"
shutil.move(name, SRC)

REF_WAV = f"{WORK}/teacher_ref.wav"   # converted: mono, 24 kHz wav
sh(f'ffmpeg -y -loglevel error -ss {REF_START_SEC} -t {REF_SECONDS} -i "{SRC}" -ac 1 -ar 24000 "{REF_WAV}"')
dur = sf.info(REF_WAV).duration
print(f"Reference clip: {dur:.1f} seconds")
if dur < 6:
    print("WARNING: shorter than 6 seconds. Use a longer clip or change REF_START_SEC / REF_SECONDS.")
TEACHER_FILES_FOR_ZIP = [REF_WAV]

from IPython.display import Audio, display
display(Audio(REF_WAV))

**Play the clip above.** It must be only the teacher, clear and continuous. If not, change `REF_START_SEC` in the config cell and re-run Step 3.

## Step 4: Generate the 20 sentences

In [ ]:
assert ACCEPT_COQUI_LICENSE, "Read the Coqui Public Model License, then set ACCEPT_COQUI_LICENSE = True in the config cell and re-run."
os.environ["COQUI_TOS_AGREED"] = "1"

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

for i, s in enumerate(SENTENCES):
    tts.tts_to_file(text=s, speaker_wav=REF_WAV, language="en", file_path=f"{OUT}/{i:02d}.wav")
    print(f"{i + 1}/{len(SENTENCES)}")

## Step 5: Listen and download the results zip

In [ ]:
from IPython.display import Audio, display
for idx in (0, 13):          # one general sentence and one technical sentence
    p = f"{OUT}/{idx:02d}.wav"
    if os.path.exists(p):
        print(f"--- sentence {idx}: {SENTENCES[idx]}")
        display(Audio(p))

RES = f"/content/{METHOD}_results"
shutil.rmtree(RES, ignore_errors=True)
os.makedirs(f"{RES}/teacher_audio", exist_ok=True)
shutil.copytree(OUT, f"{RES}/outputs")
for p in TEACHER_FILES_FOR_ZIP:
    shutil.copy(p, f"{RES}/teacher_audio/")
open(f"{RES}/sentences.txt", "w").write("\n".join(SENTENCES) + "\n")
shutil.make_archive(f"/content/{METHOD}_results", "zip", RES)
from google.colab import files
files.download(f"/content/{METHOD}_results.zip")
print("Downloaded", f"{METHOD}_results.zip", "- keep it, the compare notebook needs it.")